# 01 EDA: Tablo grain'leri

Her tablo için önce grain hipotezi, sonra duplicate testi.

In [285]:
import sys
print(sys.executable)

import seller_propensity
print(seller_propensity.__file__)

/Users/mudogan/Desktop/seller-propensity/.venv/bin/python
/Users/mudogan/Desktop/seller-propensity/src/seller_propensity/__init__.py


In [286]:
import pandas as pd
from pathlib import Path

In [287]:
def short_name(path: Path) -> str:
    """Short name for a dataset file path, removing the prefix and suffix."""
    name = path.stem
    name = name.removeprefix("olist_")
    name = name.removesuffix("_dataset")
    return name

DATA_DIR = Path.cwd().parent / "data" / "raw"
data_dict = {short_name(f): pd.read_csv(f) for f in sorted(DATA_DIR.glob("*.csv"))}

In [288]:
for name, df in data_dict.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
product_category_name_translation: (71, 2)


In [289]:
def count_duplicates(df: pd.DataFrame, keys: str | list[str]) -> int:
    """Count the number of duplicate rows in a DataFrame based on specified keys."""
    return df.duplicated(subset=keys, keep='first').sum()

## 1. orders

In [290]:
data_dict["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


**orders için Hipotez:**

1 satır = bir sipariş ve key = order_id

In [291]:
print(f"order_id: {count_duplicates(data_dict['orders'], ['order_id'])}")

order_id: 0


Hipotez doğru.

## 2. order_items

In [292]:
data_dict["order_items"].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


**order_items için Hipotez:**

İlk tahmin key = order_id. Tutmazsa 1 satır = bir siparişteki bir ürün kalemi ve key = (order_id, order_item_id)

In [293]:
print(f"order_id: {count_duplicates(data_dict['order_items'], ['order_id'])}")
print(f"order_id + order_item_id: {count_duplicates(data_dict['order_items'], ['order_id', 'order_item_id'])}")

order_id: 13984
order_id + order_item_id: 0


order_id tek başına unique değil, bir siparişte birden fazla ürün olabiliyor. order_item_id sipariş içindeki sıra numarası. seller_id de bu tabloda, seller bazlı hesaplar buradan geçecek.

## 3. order_reviews

In [294]:
data_dict["order_reviews"].head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


**order_reviews için Hipotez:**

1 satır = bir review ile bir sipariş eşleşmesi ve key = (order_id, review_id)

In [295]:
print(f"order_id: {count_duplicates(data_dict['order_reviews'], ['order_id'])}")
print(f"review_id: {count_duplicates(data_dict['order_reviews'], ['review_id'])}")
print(f"order_id + review_id: {count_duplicates(data_dict['order_reviews'], ['order_id', 'review_id'])}")

order_id: 551
review_id: 814
order_id + review_id: 0


Bir siparişin birden fazla review'u olabiliyor, bir review da birden fazla siparişe bağlı olabiliyor (N:M). Aynı review'un neden birden fazla siparişte olduğuna 8. bölümde bakıyorum.

## 4. order_payments

In [296]:
data_dict["order_payments"].head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


**order_payments için Hipotez:**

1 satır = bir ödeme kaydı ve key = (order_id, payment_sequential). Bir sipariş birden fazla ödeme yöntemiyle ödenebiliyor.

In [297]:
print(f"order_id: {count_duplicates(data_dict['order_payments'], ['order_id'])}")
print(f"order_id + payment_sequential: {count_duplicates(data_dict['order_payments'], ['order_id', 'payment_sequential'])}")

order_id: 4446
order_id + payment_sequential: 0


Hipotez doğru. Not: sipariş tutarı lazım olursa ödemeleri önce order_id bazında toplamak gerekiyor, yoksa join'de satır çoğalır.

## 5. customers

In [298]:
data_dict["customers"].head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**customers için Hipotez:**

1 satır = bir müşteri ve key = customer_id

In [299]:
print(f"customer_id: {count_duplicates(data_dict['customers'], ['customer_id'])}")
print(f"customer_unique_id: {count_duplicates(data_dict['customers'], ['customer_unique_id'])}")

customer_id: 0
customer_unique_id: 3345


In [300]:
data_dict["customers"][data_dict["customers"]["customer_unique_id"].duplicated(keep=False)].head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
13,eabebad39a88bb6f5b52376faec28612,295c05e81917928d76245e842748184d,5704,sao paulo,SP
32,2d5831cb2dff7cdefba62e950ae3dc7b,e9dd12dca17352644a959d9dea133935,42800,camacari,BA
33,b2bed119388167a954382cca36c4777f,e079b18794454de9d2be5c12b4392294,27525,resende,RJ
42,a3b0fda37bae14cf754877bed475e80c,c9158d089637ab443c78984d20da7fc0,5727,sao paulo,SP
88,3f6ede29d4c69cd3316d2035b6cec1fb,7a380cb5434e6b6b5b37d45bb99dbe8a,9890,sao bernardo do campo,SP
113,8891eb5ca0e28df961b2b5b8f3c0eb23,13ac6da47fef1b17b16f511e03b57918,2632,sao paulo,SP
116,fe6d73ac006153a398439253006e5adc,60f874c577dc096ecd52bd0bf6bd5942,36780,astolfo dutra,MG
132,f7c5afab273b47ab517e096e0219b932,b6c083700ca8c135ba9f0f132930d4e8,8686,suzano,SP


In [301]:
data_dict["customers"][data_dict["customers"]["customer_unique_id"] == "4c93744516667ad3b8f1fb645a3116a4"]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
66803,802bb9a59876a712f8380da8f297057c,4c93744516667ad3b8f1fb645a3116a4,89253,jaragua do sul,SC


Bu örnekte zip kodu farklı ama tek örnekten sonuç çıkmaz. customers ve orders ikisi de 99441 satır, customer_id orders'ta kaç kez geçiyor bakalım.

In [302]:
print(f"orders'ta customer_id: {count_duplicates(data_dict['orders'], ['customer_id'])}")
print(data_dict['customers']['customer_id'].isin(data_dict['orders']['customer_id']).all())

orders'ta customer_id: 0
True


customer_id orders ile 1:1, yani her sipariş için ayrı üretilmiş bir id. Gerçek müşteri customer_unique_id. Grain: 1 satır = bir siparişin müşteri kaydı. Müşteri bazlı hesaplarda customer_unique_id kullanılmalı.

## 6. products

In [303]:
data_dict["products"].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


**products için Hipotez:**

1 satır = bir ürün ve key = product_id

In [304]:
print(f"product_id: {count_duplicates(data_dict['products'], ['product_id'])}")

product_id: 0


Hipotez doğru.

## 7. sellers

In [305]:
data_dict["sellers"].head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


**sellers için Hipotez:**

1 satır = bir satıcı ve key = seller_id

In [306]:
print(f"seller_id: {count_duplicates(data_dict['sellers'], ['seller_id'])}")

seller_id: 0


Hipotez doğru.

## 8. Birden fazla siparişe bağlı review'lar

Hipotezim: müşteri aynı gün birden fazla sipariş verdi, tek review yazdı ve review hepsine bağlandı. Doğruysa bu siparişler aynı kişiye ve aynı güne ait olmalı.

In [307]:
review_w_order = data_dict["order_reviews"].merge(data_dict["orders"], how='left', on='order_id')
review_w_customer = review_w_order.merge(data_dict["customers"], how='left', on='customer_id')

# join keys are unique on the right side, row count should stay the same
print(len(data_dict["order_reviews"]) == len(review_w_order) == len(review_w_customer))

review_w_customer['order_purchase_date'] = pd.to_datetime(review_w_customer['order_purchase_timestamp']).dt.normalize()

cols = ['order_id', 'customer_id', 'customer_unique_id', 'order_purchase_date', 'order_purchase_timestamp']
shared_reviews = review_w_customer.groupby('review_id')[cols].nunique().query('order_id > 1')

print(f"birden fazla siparişe bağlı review: {len(shared_reviews)}")
for col in cols[1:]:
    print(f"tek {col}: {(shared_reviews[col] == 1).mean():.1%}")

True
birden fazla siparişe bağlı review: 789
tek customer_id: 0.0%
tek customer_unique_id: 100.0%
tek order_purchase_date: 94.8%
tek order_purchase_timestamp: 32.2%


Hepsi aynı customer_unique_id'ye ait, yüzde 95'i aynı gün ama sadece yüzde 32'si aynı saniye. Yani sistem hatası değil, aynı gün verilen siparişlere tek review yazılmış. customer_id ile bakınca yüzde 0 çıkıyor çünkü customer_id her siparişte farklı.

Not: seller bazında review skoru hesaplarken aynı review iki kez sayılmasın.

## Özet

| Tablo | 1 satır | Key |
|---|---|---|
| orders | sipariş | order_id |
| order_items | siparişteki ürün kalemi | order_id, order_item_id |
| order_reviews | review ve sipariş eşleşmesi | order_id, review_id |
| order_payments | ödeme kaydı | order_id, payment_sequential |
| customers | siparişin müşteri kaydı | customer_id |
| products | ürün | product_id |
| sellers | satıcı | seller_id |